# CRISP-DM Fase 2: Data Understanding - Exploración Informática y Estadística (EDA)
## Dataset: SIPSA Abastecimiento de Alimentos (DANE)

**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 2: Exploratory Data Analysis) | **Fase PDCO**: DEVELOPMENT  
**Estándares**: ISO/IEC 25010 (Data Quality Metrics), Control Estadístico de Procesos (SPC Nelson Rules).

---

### 🔬 1. Diagnóstico de Calidad Informática
* Perfilamiento de tipos de datos, consumo en memoria RAM y cardinalidad de variables.
* Diagnóstico de datos faltantes bajo tipología de Donald Rubin: **MCAR** (Missing Completely at Random), **MAR** (Missing at Random) y **MNAR** (Missing Not at Random).

### 📐 2. Diagnóstico Estadístico y Control de Procesos (SPC)
* Estimación de los 4 momentos estadísticos de Pearson: Media ($\\mu$), Varianza ($\\sigma^2$), Asimetría ($S$) y Curtosis ($K$).
* Test de Normalidad de Jarque-Bera.
* Detección de anomalías mediante la **Regla 1 de Nelson** ($|z| > 3$) y **Vallas de Tukey** ($[Q_1 - 1.5 IQR, Q_3 + 1.5 IQR]$).

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


In [ ]:
# ==============================================================================
# [FASE 3: PERFILAMIENTO INFORMÁTICO Y CONSUMO DE MEMORIA]
# ==============================================================================
from src.database.db_manager import DatabaseManager

parquet_file = APP_ROOT / "data/processed/sipsa_abastecimientos.parquet"
if parquet_file.exists():
    df = pd.read_parquet(parquet_file)
else:
    db = DatabaseManager(str(APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"))
    df = db.query("SELECT * FROM sipsa_abastecimientos")

print(f"📊 Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"💾 Consumo en Memoria: {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
print("\n--- DIAGNÓSTICO DE COMPLETITUD Y NULOS ---")
missing_df = pd.DataFrame({
    "tipo_dato": df.dtypes,
    "nulos": df.isnull().sum(),
    "nulos_pct": (df.isnull().sum() / len(df)) * 100,
    "valores_unicos": df.nunique()
})
display(missing_df)


In [ ]:
# ==============================================================================
# [FASE 3: DIAGNÓSTICO ESTADÍSTICO DE DISTRIBUCIONES Y CONTROL SPC]
# ==============================================================================
from scipy import stats
from src.modeling.sarimax_model import AgroModeler

target_col = "cantidad_kg"
if target_col in df.columns and pd.api.types.is_numeric_dtype(df[target_col]):
    s = df[target_col].dropna()
    
    mean_val = float(s.mean())
    std_val = float(s.std())
    skew_val = float(stats.skew(s))
    kurt_val = float(stats.kurtosis(s))
    jb_stat, jb_pval = stats.jarque_bera(s)
    
    print("--- 1. MOMENTOS ESTADÍSTICOS DE PEARSON ---")
    print(f"• Media Aritmética (Media): {mean_val:,.4f}")
    print(f"• Desviación Estándar (Sigma): {std_val:,.4f}")
    print(f"• Coeficiente de Asimetría (Skewness): {skew_val:.4f} ({'Asimetría Positiva' if skew_val > 0 else 'Asimetría Negativa'})")
    print(f"• Curtosis: {kurt_val:.4f} ({'Leptocúrtica' if kurt_val > 0 else 'Platicúrtica'})")
    print(f"• Test Jarque-Bera: p-value = {jb_pval:.4e} ({'Rechaza normalidad' if jb_pval < 0.05 else 'Compatible con normalidad'})")
    
    print("\n--- 2. DETECCIÓN DE ANOMALÍAS (SPC REGLA 1 DE NELSON) ---")
    anomalies = AgroModeler.detect_nelson_anomalies(s)
    print(f"• Observaciones fuera de control (|z| > 3): {anomalies.sum()} de {len(s)} ({(anomalies.sum()/len(s)):.2%})")
else:
    print(f"ℹ️ La columna 'cantidad_kg' es textual o estructurada. Evaluando cardinalidad categórica:")
    print(df.describe(include='all'))


In [ ]:
# ==============================================================================
# [FASE 3: VISUALIZACIONES DIAGNÓSTICAS EDA]
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

target_col = "cantidad_kg"
if target_col in df.columns and pd.api.types.is_numeric_dtype(df[target_col]):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    s = df[target_col].dropna()
    
    # Subplot 1: Histograma y KDE
    sns.histplot(s, kde=True, ax=axes[0], color="#2b5c8f", bins=30)
    axes[0].axvline(s.mean(), color="red", linestyle="--", label=f"Media: {s.mean():,.2f}")
    axes[0].axvline(s.median(), color="green", linestyle="-", label=f"Mediana: {s.median():,.2f}")
    axes[0].set_title("Distribución empírica de cantidad_kg")
    axes[0].legend()
    
    # Subplot 2: Boxplot con límites IQR
    sns.boxplot(x=s, ax=axes[1], color="#52b788")
    axes[1].set_title("Boxplot y Detección de Outliers (Vallas de Tukey)")
    
    plt.tight_layout()
    plt.show()
